# 13 — LoRA and PEFT: Practical Fine-Tuning

**Network LLM Engineering — Part III — Adaptation**

### Learning goals
- Understand low-rank adapters
- Train a small networking adapter
- Reason about rank, alpha, target modules and adapter storage

In [ ]:
%pip install -q transformers==5.14.1 datasets==5.0.1 accelerate==1.14.0 peft==0.20.0 trl==1.10.0 sentence-transformers==5.7.0 pandas matplotlib scikit-learn requests jsonschema

In [ ]:
from pathlib import Path

def find_root():
    for p in [Path.cwd(), Path.cwd().parent, Path("/content/network_llm_engineering_course")]:
        if (p / "data" / "glossary.csv").exists():
            return p
    raise FileNotFoundError("Run from the extracted network_llm_engineering_course folder.")

ROOT = find_root()
DATA = ROOT / "data"
print("Course root:", ROOT)

## LoRA intuition

Instead of updating a large weight matrix `W`, LoRA learns a low-rank update approximately `B @ A`.
The base model stays frozen; only small adapter matrices train.

Advantages:
- far fewer trainable parameters,
- smaller checkpoints,
- easier specialist adapters,
- lower training memory.

Tradeoff: it is still real model training and can still overfit or learn bad behavior.

In [ ]:
d = 1024
for r in [4,8,16,32,64]:
    full = d*d
    lora = d*r + r*d
    print(f"rank={r:2d}: LoRA params={lora:7,d} ({100*lora/full:5.2f}% of one full matrix)")

In [ ]:
# GPU lab: construct and optionally train the adapter.
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen3-0.6B"
train = load_dataset("json", data_files=str(DATA/"network_sft_train.jsonl"), split="train")
eval_ds = load_dataset("json", data_files=str(DATA/"network_sft_eval.jsonl"), split="train")
drop = [c for c in train.column_names if c not in {"prompt","completion"}]
train = train.remove_columns(drop)
drop2 = [c for c in eval_ds.column_names if c not in {"prompt","completion"}]
eval_ds = eval_ds.remove_columns(drop2)

peft_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

args = SFTConfig(
    output_dir=str(ROOT/"artifacts"/"network-lora"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=3,
    max_length=512,
    completion_only_loss=True,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
)
print("Configuration ready. The next cell performs training.")

In [ ]:
if not torch.cuda.is_available():
    print("GPU not detected. Switch Colab to a GPU runtime, then rerun this cell.")
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype, device_map="auto")
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train,
        eval_dataset=eval_ds,
        peft_config=peft_cfg,
        processing_class=tokenizer,
    )
    trainer.model.print_trainable_parameters()
    result = trainer.train()
    trainer.save_model(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    print("Saved adapter:", args.output_dir)

### Exercise

Run rank 8 and rank 32 with the same seed/data. Compare held-out task scores, adapter size, and signs of overfitting.